<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Perception.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install nuscenes-devkit pyquaternion pyyaml tqdm

In [ ]:
import os, yaml, json, random
import numpy as np

# CHANGE THIS to your actual folder in Drive
DATA_ROOT = "/content/drive/MyDrive/datasets/nuscenes"   # contains v1.0-mini/
VERSION = "v1.0-mini"
DATASET_ROOT = os.path.join(DATA_ROOT, VERSION)

assert os.path.exists(DATASET_ROOT), f"Not found: {DATASET_ROOT}"

CFG = {
  "version": VERSION,
  "dataroot": DATASET_ROOT,
  "bev": {
    "x_min": -25.0, "x_max":  25.0,
    "y_min": -25.0, "y_max":  25.0,
    "resolution": 0.25,
    "z_min": -3.0, "z_max":  2.0,
  },
  "export": {
    "out_dir": "/content/drive/MyDrive/bev_dataset",
    "max_scenes": 3,              # start small
    "max_frames_per_scene": 60,   # start small
    "seed": 42
  }
}

print("Dataset root:", DATASET_ROOT)
print("Export out_dir:", CFG["export"]["out_dir"])

In [ ]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version=CFG["version"], dataroot=CFG["dataroot"], verbose=True)
print("Scenes:", len(nusc.scene))

In [ ]:
import os
import numpy as np
from pyquaternion import Quaternion
from nuscenes.utils.data_classes import LidarPointCloud
from nuscenes.utils.geometry_utils import transform_matrix

def load_lidar_xyz_ego(nusc, sample_token: str, dataroot: str) -> np.ndarray:
    """
    Returns xyz points in EGO frame at the lidar timestamp. Shape: (N, 3)
    """
    sample = nusc.get("sample", sample_token)
    lidar_token = sample["data"]["LIDAR_TOP"]
    sd = nusc.get("sample_data", lidar_token)

    # nuScenes uses 'filename' (not 'file_name') :contentReference[oaicite:1]{index=1}
    lidar_path = os.path.join(dataroot, sd["filename"])
    pc = LidarPointCloud.from_file(lidar_path)

    # Lidar sensor -> Ego frame
    cs = nusc.get("calibrated_sensor", sd["calibrated_sensor_token"])
    pc.transform(transform_matrix(cs["translation"], Quaternion(cs["rotation"]), inverse=False))

    xyz = pc.points[:3, :].T  # (N, 3)
    return xyz

def rasterize_bev_occupancy(xyz_ego: np.ndarray, bev_cfg: dict, min_points_per_cell: int = 1) -> np.ndarray:
    x_min, x_max = bev_cfg["x_min"], bev_cfg["x_max"]
    y_min, y_max = bev_cfg["y_min"], bev_cfg["y_max"]
    res = bev_cfg["resolution"]
    z_min = bev_cfg.get("z_min", None)
    z_max = bev_cfg.get("z_max", None)

    x = xyz_ego[:, 0]
    y = xyz_ego[:, 1]
    z = xyz_ego[:, 2]

    mask = (x >= x_min) & (x < x_max) & (y >= y_min) & (y < y_max)
    if z_min is not None:
        mask &= (z >= z_min)
    if z_max is not None:
        mask &= (z < z_max)

    x = x[mask]
    y = y[mask]

    H = int((x_max - x_min) / res)
    W = int((y_max - y_min) / res)

    counts = np.zeros((H, W), dtype=np.uint16)

    x_idx = ((x - x_min) / res).astype(np.int32)
    y_idx = ((y - y_min) / res).astype(np.int32)

    x_idx = np.clip(x_idx, 0, H - 1)
    y_idx = np.clip(y_idx, 0, W - 1)

    np.add.at(counts, (x_idx, y_idx), 1)
    occ = (counts >= min_points_per_cell).astype(np.uint8)
    return occ

In [ ]:
import matplotlib.pyplot as plt
import random

scene = random.choice(nusc.scene)
token = scene["first_sample_token"]
xyz = load_lidar_xyz_ego(nusc, token, CFG["dataroot"])
occ = rasterize_bev_occupancy(xyz, CFG["bev"])

print("xyz:", xyz.shape, "occ:", occ.shape, "occupied cells:", int(occ.sum()))

plt.figure(figsize=(6,6))
plt.imshow(occ.T, origin="lower")
plt.title("BEV Occupancy (ego frame, symmetric)")
plt.xlabel("x bins")
plt.ylabel("y bins")
plt.show()

In [ ]:
from tqdm import tqdm
import os, json

def iter_scene_samples(nusc, scene_record, max_frames=None):
    token = scene_record["first_sample_token"]
    count = 0
    while token:
        yield token
        count += 1
        if max_frames is not None and count >= max_frames:
            break
        sample = nusc.get("sample", token)
        token = sample["next"]

def export_bev_dataset(nusc, cfg):
    out_dir = cfg["export"]["out_dir"]
    os.makedirs(out_dir, exist_ok=True)

    # Save meta/config
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    scenes = list(nusc.scene)
    random.seed(cfg["export"]["seed"])
    random.shuffle(scenes)
    scenes = scenes[: cfg["export"]["max_scenes"]]

    train_dir = os.path.join(out_dir, "train")
    os.makedirs(train_dir, exist_ok=True)

    for scene in scenes:
        scene_name = scene["name"]
        scene_out = os.path.join(train_dir, scene_name)
        os.makedirs(scene_out, exist_ok=True)

        tokens = list(iter_scene_samples(nusc, scene, max_frames=cfg["export"]["max_frames_per_scene"]))
        for token in tqdm(tokens, desc=f"Export {scene_name}"):
            xyz = load_lidar_xyz_ego(nusc, token, cfg["dataroot"])
            occ = rasterize_bev_occupancy(xyz, cfg["bev"], min_points_per_cell=1)
            np.save(os.path.join(scene_out, f"{token}_occ.npy"), occ)

    return out_dir

out_dir = export_bev_dataset(nusc, CFG)
print("Export complete:", out_dir)

In [ ]:
import glob, random
import numpy as np
import matplotlib.pyplot as plt

files = glob.glob(os.path.join(CFG["export"]["out_dir"], "train", "*", "*_occ.npy"))
print("Exported files:", len(files))

path = random.choice(files)
occ = np.load(path)

plt.figure(figsize=(6,6))
plt.imshow(occ.T, origin="lower")
plt.title(f"Exported BEV\n{os.path.basename(path)}")
plt.show()